Project: PetCare AI – Pet Grooming Assistant 🐾
Imagine you own a pet grooming business. You want an AI assistant on your website that can answer customer questions and collect leads from people interested in booking grooming services.

This is a good beginner project because it involves one LLM, one custom tool, and one CSV file.


In [ ]:
import os
import json
import csv

from groq import Groq
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv()
groq_api_key = os.getenv('GROQ_API_KEY')
groq = OpenAI(base_url = "https://api.groq.com/openai/v1",api_key=groq_api_key)
MODEL = "openai/gpt-oss-120b"

print("Groq connected successfully!")

In [28]:
business_info = """
You are PetCare AI, the virtual assistant
for PetCare Grooming Services.

BUSINESS INFORMATION:

Business Name: PetCare Grooming Services

Location: Vadodara, Gujarat.

SERVICES:

1. Dog Bathing - Rs. 500
2. Dog Haircut - Rs. 800
3. Cat Grooming - Rs. 700
4. Nail Trimming - Rs. 200
5. Full Pet Grooming - Rs. 1200

OPENING HOURS:

Monday to Saturday:
9 AM to 7 PM

Sunday:
Closed.

INSTRUCTIONS:

1. Answer customer questions using the
   business information provided.

2. Be friendly and professional.

3. Never invent services or prices.

4. If a customer is interested in booking
   a service, ask for their name and email.

5. Only record customer information after
   they explicitly agree to share it for
   booking follow-up.

6. Once you have the required information
   and consent, use the save_customer tool.

7. Never claim a booking is confirmed.
   We only collect booking inquiries.
"""

In [29]:
def save_customer(name, email, service):

    if "@" not in email or "." not in email.split("@")[-1]:
        return "Invalid email address."

    file_exists = os.path.exists("customers.csv")

    with open("customers.csv", "a", newline="") as file:

        writer = csv.writer(file)

        if not file_exists:
            writer.writerow(["Name", "Email", "Service"])

        writer.writerow([name, email, service])

    return "Customer inquiry successfully recorded!"

In [30]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "save_customer",
            "description": """
            Save customer details when someone wants
            to book a grooming service.

            Only call this tool after collecting
            the customer's name, email, service,
            and explicit consent to record the inquiry.
            """,
            "parameters": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string",
                        "description": "Customer name"
                    },
                    "email": {
                        "type": "string",
                        "description": "Customer email address"
                    },
                    "service": {
                        "type": "string",
                        "description": "Interested grooming service"
                    }
                },
                "required": ["name", "email", "service"]
            }
        }
    }
]

In [31]:
available_tools = {
    "save_customer": save_customer
}

In [32]:
messages = [
    {
        "role": "system",
        "content": business_info
    }
]

In [ ]:
def chat(message, history):


    clean_history = [
        {
            "role": item["role"],
            "content": item["content"]
        }
        for item in history
        if item["role"] in ["user", "assistant"]
    ]


    messages = [
        {"role": "system", "content": business_info}
    ] + clean_history + [
        {"role": "user", "content": message}
    ]

    # First LLM call

    response = groq.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )

    response_message = response.choices[0].message

    # Check for tool calls

    if response_message.tool_calls:

        messages.append(response_message)

        for tool_call in response_message.tool_calls:

            function_name = tool_call.function.name

            arguments = json.loads(
                tool_call.function.arguments
            )

            print("Tool Called:", function_name)
            print("Arguments:", arguments)

            function_to_call = available_tools[function_name]

            result = function_to_call(**arguments)

            print("Tool Result:", result)

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result
            })

        # Second LLM call

        response = groq.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools
        )

    return response.choices[0].message.content

In [ ]:
response = chat("What is the cost for dog grooming?", [])

print(response)

In [ ]:
result = save_customer(
    name="Shannu",
    email="shannu@gmail.com",
    service="Dog Bathing"
)

print(result)

In [ ]:
import gradio as gr

demo = gr.ChatInterface(
    fn=chat
)

demo.launch()